# Part 6: Callbacks

In PyTorch you wrote code inside your loop to keep the best weights and to stop early. In Keras you do not touch the loop. You hand `fit` a list of **callbacks**, objects with hooks that `fit` calls at the start and end of every epoch and batch.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow start-up messages

import tempfile
from pathlib import Path

import keras

from keras_net import build_keras_net

import numpy as np

from churn_data import make_splits

splits = make_splits(42)
x_train, y_train = splits.train.x.numpy(), splits.train.y.numpy()
x_val, y_val = splits.val.x.numpy(), splits.val.y.numpy()
print(x_train.shape, y_train.mean().round(3))

folder = Path(tempfile.mkdtemp())

## The four callbacks you will use most

| Callback | What it does |
|---|---|
| `EarlyStopping` | stops when a monitored value stops improving, and can restore the best weights |
| `ModelCheckpoint` | saves the model, with `save_best_only=True` it saves only on improvement |
| `ReduceLROnPlateau` | lowers the learning rate when progress stalls |
| `CSVLogger` | writes every epoch's numbers to a file |

You can also write your own by subclassing `keras.callbacks.Callback`.

In [ ]:
class PrintEvery(keras.callbacks.Callback):
    """A custom callback: print the validation loss every ten epochs."""

    def on_epoch_end(self, epoch, logs=None):
        if epoch % 10 == 0:
            print(f"epoch {epoch + 1:3d}  val_loss {logs['val_loss']:.4f}")


callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(folder / "best.keras", monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
    keras.callbacks.CSVLogger(folder / "history.csv"),
    PrintEvery(),
]

keras.utils.set_random_seed(42)
model = build_keras_net(9)
model.compile(
    optimizer=keras.optimizers.Adam(3e-3),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
)
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=200, batch_size=64, verbose=0,
    callbacks=callbacks,
)

We asked for 200 epochs. Early stopping decides when to end.

In [ ]:
val = history.history["val_loss"]
print("epochs actually run:", len(val))
print("best epoch:", int(np.argmin(val)) + 1, " best val loss:", round(min(val), 4), " last val loss:", round(val[-1], 4))
print("learning rates seen:", sorted({round(float(v), 5) for v in history.history["learning_rate"]}, reverse=True))
print("files written:", sorted(p.name for p in folder.iterdir()))

With `patience=10`, training ran for 33 epochs, because the best epoch was 23 and ten more passed with no improvement. `restore_best_weights=True` put the weights from epoch 23 back, so the model in memory is the best one and not the last. `ReduceLROnPlateau` halved the learning rate twice near the end.

## Loading what was saved

`ModelCheckpoint` wrote a `.keras` file. It holds the architecture, the weights and the optimizer state in one place, so loading needs no class definition.

In [ ]:
loaded = keras.models.load_model(folder / "best.keras")
print("loaded model:", round(float(loaded.evaluate(x_val, y_val, verbose=0)), 4))
print("model in memory:", round(float(model.evaluate(x_val, y_val, verbose=0)), 4))

model.save_weights(folder / "weights.weights.h5")  # weights only, like a state dict

Compare with PyTorch. There, saving the architecture is your code's job and `load_state_dict` restores numbers into a model you rebuilt. In Keras, `model.save` stores both, and `save_weights` is the equivalent of a state dict.

## Your turn

1. Change `monitor` to `val_auc` (add the AUC metric in `compile`) and set `mode="max"`. Does the best epoch change?
2. Add `keras.callbacks.TerminateOnNaN()` to the list. When would it save you time?